# Proyek Akhir: Pengembangan dan Pengoperasian Sistem Machine Learning

**Nama:** sintiawati

**Username Dicoding:** sintiawati

**Repo:** [chicago-taxi-mlops](https://github.com/sintiasnn/chicago-taxi-mlops)

**Dataset:** Chicago Taxi Trips

**Persoalan:** Memprediksi apakah seorang penumpang taxi akan memberikan tip atau tidak berdasarkan data perjalanan.

---

## 1. Informasi Dataset

Dataset yang digunakan adalah **Chicago Taxi Trips** yang merupakan dataset publik dari kota Chicago, Illinois, Amerika Serikat. Dataset ini berisi informasi tentang perjalanan taksi yang dilakukan di kota Chicago.

### Fitur-fitur dalam dataset:
| Fitur | Tipe | Deskripsi |
|-------|------|-----------|
| trip_start_month | Integer | Bulan dimulainya perjalanan |
| trip_start_hour | Integer | Jam dimulainya perjalanan |
| trip_start_day | Integer | Hari dimulainya perjalanan |
| trip_start_timestamp | Integer | Timestamp dimulainya perjalanan |
| pickup_latitude | Float | Latitude lokasi penjemputan |
| pickup_longitude | Float | Longitude lokasi penjemputan |
| dropoff_latitude | Float | Latitude lokasi penurunan |
| dropoff_longitude | Float | Longitude lokasi penurunan |
| trip_miles | Float | Jarak perjalanan (mil) |
| trip_seconds | Float | Durasi perjalanan (detik) |
| fare | Float | Tarif perjalanan |
| tips | Float | Jumlah tip (label/target) |
| payment_type | String | Metode pembayaran |
| company | String | Perusahaan taksi |
| pickup_community_area | Integer | Area komunitas penjemputan |
| dropoff_community_area | Integer | Area komunitas penurunan |

## 2. Persoalan yang Ingin Diselesaikan

### Latar Belakang
Dalam industri transportasi, memahami faktor-faktor yang memengaruhi pemberian tip oleh penumpang dapat membantu pengemudi dan perusahaan taksi dalam meningkatkan kualitas layanan dan pendapatan.

### Problem Statement
Membangun sistem machine learning yang dapat memprediksi apakah seorang penumpang akan memberikan tip berdasarkan karakteristik perjalanan taksi.

### Tujuan
- Mengembangkan model prediksi yang akurat untuk memprediksi pemberian tip
- Mengimplementasikan MLOps pipeline end-to-end menggunakan TFX
- Menjalankan sistem di cloud dengan monitoring menggunakan Prometheus

## 3. Solusi Machine Learning

### Pendekatan
Masalah ini adalah **klasifikasi biner** (binary classification) dimana targetnya adalah apakah penumpang memberikan tip (`tips > 0`) atau tidak.

### Target
- **1**: Penumpang memberikan tip (tips > 0)
- **0**: Penumpang tidak memberikan tip (tips = 0)

### Metode Pengolahan Data
1. **Scaling numerik**: Fitur numerik (trip_miles, fare, trip_seconds) di-scale menggunakan Z-score
2. **Feature engineering**: Membuat fitur baru `trip_speed` (trip_miles / trip_seconds)
3. **Encoding kategorikal**: Fitur kategorikal di-encode menggunakan vocab mapping
4. **Label biner**: Kolom `tips` dikonversi menjadi label biner (0/1)

### Arsitektur Model
- Input layer untuk fitur numerik dan kategorikal
- Embedding layer untuk fitur kategorikal
- Dense layers: [64, 32] dengan aktivasi ReLU
- Dropout 20% untuk regularisasi
- Output layer: 1 neuron dengan sigmoid activation

### Metrik Evaluasi
- **Binary Accuracy**: Akurasi prediksi
- **AUC**: Performa model secara keseluruhan
- **Precision**: Ketepatan prediksi positif
- **Recall**: Sensitivitas model

## 4. Import Library dan Setup Environment

In [ ]:
import os
import sys
import logging

import tensorflow as tf
import pandas as pd

from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

logging.basicConfig(level=logging.INFO)

print(f"TensorFlow version: {tf.__version__}")
print(f"Python version: {sys.version}")

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Project directory: {PROJECT_DIR}")

## 5. Eksplorasi Data

In [ ]:
DATA_PATH = "data/data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Info:")
df.info()

In [ ]:
print("Descriptive Statistics:")
df.describe()

In [ ]:
print("Label Distribution (tips > 0):")
df["tips_binary"] = (df["tips"] > 0).astype(int)
print(df["tips_binary"].value_counts())
print(f"\nPercentage:")
print(df["tips_binary"].value_counts(normalize=True) * 100)

In [ ]:
print("Missing Values:")
df.isnull().sum()

## 6. Menjalankan TFX Pipeline

Pipeline akan menjalankan seluruh komponen:
1. **ExampleGen**: Memuat dan membagi dataset
2. **StatisticsGen**: Menghasilkan statistik data
3. **SchemaGen**: Membuat skema data
4. **ExampleValidator**: Memvalidasi data
5. **Transform**: Mentransformasi data
6. **Trainer**: Melatih model
7. **Resolver**: Mendapatkan model terbaik sebelumnya
8. **Evaluator**: Mengevaluasi model
9. **Pusher**: Mempush model ke serving

In [ ]:
%%time
from taxi_pipeline.pipeline_def import create_pipeline

pipeline = create_pipeline()
BeamDagRunner().run(pipeline)
print("Pipeline execution completed!")

## 7. Evaluasi Hasil Pipeline

In [ ]:
output_dir = "output"
if os.path.exists(output_dir):
    print("Pipeline output directories:")
    for root, dirs, files in os.walk(output_dir):
        level = root.replace(output_dir, "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for file in files[:5]:
            print(f"{subindent}{file}")
else:
    print("Output directory not found.")

In [ ]:
serving_model_dir = "serving_model"
if os.path.exists(serving_model_dir):
    saved_models = [
        d for d in os.listdir(serving_model_dir) 
        if os.path.isdir(os.path.join(serving_model_dir, d))
    ]
    print(f"Saved models: {saved_models}")
    if saved_models:
        latest = max(saved_models, key=lambda d: os.path.getmtime(
            os.path.join(serving_model_dir, d)
        ))
        print(f"Latest model: {latest}")
else:
    print("Serving model directory not found.")

## 8. Performa Model

Model yang dihasilkan dievaluasi menggunakan metrik:
- **Binary Accuracy**: Mengukur persentase prediksi yang benar
- **AUC**: Mengukur kemampuan model membedakan kelas positif dan negatif
- **Precision**: Proporsi prediksi positif yang benar
- **Recall**: Proporsi data positif yang berhasil dideteksi

Model yang memenuhi threshold evaluasi akan dianggap "blessed" dan siap di-push ke serving.

## 9. Model Deployment

Model akan di-deploy menggunakan **Docker** dan dijalankan di platform **Railway** (cloud).

### Opsi Deployment:
- **Railway**: Platform cloud yang mudah digunakan
- **Docker**: Containerization untuk konsistensi environment
- **Flask API**: REST API untuk model serving

### Langkah Deployment:
1. Build Docker image
2. Push image ke Railway
3. Jalankan container dengan exposed port 8080

### Endpoint API:
- `GET /` - Health check
- `GET /v1/models/taxi-model` - Informasi model
- `POST /v1/models/taxi-model:predict` - Prediksi

## 10. Monitoring dengan Prometheus

Sistem monitoring diimplementasikan menggunakan **Prometheus** dengan metrik:
- Request count
- Request latency
- Prediction distribution
- System resource usage

### Konfigurasi Monitoring:
- Prometheus container berjalan dengan port 9090
- Scrape interval: 10 detik
- Target: Aplikasi ML serving (port 8080)

## 10. Prediksi dengan TF Serving

Setelah model berhasil di-deploy ke Railway menggunakan TF Serving, kita dapat melakukan prediksi dengan mengirim request ke endpoint TF Serving.

Model menggunakan input serialized **tf.Example** yang di-encode ke base64, sesuai format standar TF Serving.

In [ ]:
import base64
import requests
import tensorflow as tf

# URL TF Serving di Railway
TF_SERVING_URL = "https://chicago-taxi-mlops-production.up.railway.app"

# Sample 1: Perjalanan dengan Credit Card (expected: high probability of tip)
feature_cc = {
    "trip_miles": tf.train.Feature(float_list=tf.train.FloatList(value=[2.5])),
    "fare": tf.train.Feature(float_list=tf.train.FloatList(value=[8.5])),
    "trip_seconds": tf.train.Feature(int64_list=tf.train.Int64List(value=[600])),
    "trip_start_timestamp": tf.train.Feature(int64_list=tf.train.Int64List(value=[1437076800])),
    "payment_type": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"Credit Card"])),
    "company": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"Taxi Affiliation Services"])),
    "trip_start_hour": tf.train.Feature(int64_list=tf.train.Int64List(value=[14])),
    "trip_start_day": tf.train.Feature(int64_list=tf.train.Int64List(value=[3])),
    "trip_start_month": tf.train.Feature(int64_list=tf.train.Int64List(value=[5])),
}
example_cc = tf.train.Example(features=tf.train.Features(feature=feature_cc))
serialized_cc = base64.b64encode(example_cc.SerializeToString()).decode("utf-8")

# Sample 2: Perjalanan dengan Cash (expected: lower probability of tip)
feature_cash = {
    "trip_miles": tf.train.Feature(float_list=tf.train.FloatList(value=[1.2])),
    "fare": tf.train.Feature(float_list=tf.train.FloatList(value=[5.85])),
    "trip_seconds": tf.train.Feature(int64_list=tf.train.Int64List(value=[180])),
    "trip_start_timestamp": tf.train.Feature(int64_list=tf.train.Int64List(value=[1382319000])),
    "payment_type": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"Cash"])),
    "company": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"Taxi Affiliation Services"])),
    "trip_start_hour": tf.train.Feature(int64_list=tf.train.Int64List(value=[1])),
    "trip_start_day": tf.train.Feature(int64_list=tf.train.Int64List(value=[2])),
    "trip_start_month": tf.train.Feature(int64_list=tf.train.Int64List(value=[10])),
}
example_cash = tf.train.Example(features=tf.train.Features(feature=feature_cash))
serialized_cash = base64.b64encode(example_cash.SerializeToString()).decode("utf-8")

payload = {"instances": [
    {"examples": {"b64": serialized_cc}},
    {"examples": {"b64": serialized_cash}},
]}

print("Sending prediction request...")
response = requests.post(
    f"{TF_SERVING_URL}/v1/models/taxi-model:predict",
    json=payload
)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print(f"Result: {result}")
    for i, pred in enumerate(result["predictions"]):
        prob = pred[0]
        print(f"Sample {i+1}: probability={prob:.4f}, prediction={1 if prob > 0.5 else 0}")
else:
    print(f"Error: {response.text}")

## 11. Kesimpulan

Proyek ini berhasil mengimplementasikan **MLOps pipeline end-to-end** menggunakan:
1. **TFX** untuk membangun pipeline machine learning yang terstruktur
2. **Apache Beam** sebagai pipeline orchestrator
3. **Docker** untuk containerization
4. **Railway** sebagai cloud platform
5. **Prometheus** untuk monitoring sistem

Dengan pipeline ini, proses pengembangan dan pengoperasian model machine learning dapat dilakukan secara otomatis, terukur, dan mudah dimonitoring.